# 0. Problem
## 1321. Restaurant Growth — Medium
Aggregate revenue per day, then return 7-day rolling amount and average for dates with a complete 7-day window.

Official: https://leetcode.com/problems/restaurant-growth/

# 1. Setup

In [ ]:
import pandas as pd
customer_rows=[(1,"Jhon","2019-01-01",100),(2,"Daniel","2019-01-02",110),(3,"Jade","2019-01-03",120),(4,"Khaled","2019-01-04",130),(5,"Winston","2019-01-05",110),(6,"Elvis","2019-01-06",140),(7,"Anna","2019-01-07",150),(8,"Maria","2019-01-08",80),(9,"Jaze","2019-01-09",110),(1,"Jhon","2019-01-10",130),(3,"Jade","2019-01-10",150)]
customer_pd=pd.DataFrame(customer_rows,columns=["customer_id","name","visited_on","amount"])
customer_pd["visited_on"]=pd.to_datetime(customer_pd["visited_on"])
customer_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
customer_spark=spark.createDataFrame(customer_rows,["customer_id","name","visited_on","amount"]).withColumn("visited_on",F.to_date("visited_on"))
customer_spark.createOrReplaceTempView("Customer")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH daily AS (
  SELECT visited_on,SUM(amount) AS daily_amount
  FROM Customer
  GROUP BY visited_on
),
rolling AS (
  SELECT visited_on,
         SUM(daily_amount) OVER(ORDER BY visited_on ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS amount,
         ROUND(AVG(daily_amount) OVER(ORDER BY visited_on ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS average_amount,
         ROW_NUMBER() OVER(ORDER BY visited_on) AS rn
  FROM daily
)
SELECT visited_on,amount,average_amount
FROM rolling
WHERE rn>=7
ORDER BY visited_on
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
daily_pd=(customer_pd.groupby("visited_on",as_index=False).agg(daily_amount=("amount","sum")).sort_values("visited_on").reset_index(drop=True))
daily_pd["amount"]=daily_pd["daily_amount"].rolling(7).sum()
daily_pd["average_amount"]=daily_pd["daily_amount"].rolling(7).mean().round(2)
result_pd=daily_pd.loc[daily_pd["amount"].notna(),["visited_on","amount","average_amount"]].reset_index(drop=True)
result_pd

# 4. PySpark Solution

In [ ]:
daily=customer_spark.groupBy("visited_on").agg(F.sum("amount").alias("daily_amount"))
w=Window.orderBy("visited_on").rowsBetween(-6,0)
rn_w=Window.orderBy("visited_on")
result_spark=(daily.withColumn("amount",F.sum("daily_amount").over(w)).withColumn("average_amount",F.round(F.avg("daily_amount").over(w),2)).withColumn("rn",F.row_number().over(rn_w)).filter(F.col("rn")>=7).select("visited_on","amount","average_amount").orderBy("visited_on"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| pre-aggregate daily | `GROUP BY day` | `.groupby()` | `.groupBy()` |
| rolling 7 | `ROWS BETWEEN 6 PRECEDING` | `.rolling(7)` | `.rowsBetween(-6,0)` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Customer

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: customer_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: customer_spark